# 03 — Manufacturing KPIs

The non-reward, operations-oriented metrics computed at episode end:
MTTR, fleet availability, throughput, total breakdowns / repairs,
finished products, etc.  These are the indicators a *factory
operator* would care about.

In [ ]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

In [ ]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

## Side-by-side KPI evolution

In [ ]:
metric_cols = group_columns(ep, "metric")
metric_cols = [c for c in metric_cols if ep[c].notna().any()]
print(f"{len(metric_cols)} episode metric columns")

if metric_cols:
    n = len(metric_cols)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.8 * nrows), squeeze=False)
    train = ep[ep["phase"] == "train"]
    for ax, col in zip(axes.flat, metric_cols):
        ax.plot(train["episode"], train[col], lw=1, color="C0", alpha=0.4)
        ax.plot(train["episode"], rolling_mean(train[col]), lw=2, color="C0")
        ax.set_title(col.split('/', 1)[-1])
        ax.set_xlabel("episode")
    for ax in axes.flat[len(metric_cols):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

## Sim time vs wall-clock time

In [ ]:
if "wall_clock_seconds" in ep.columns and "sim_time_end" in ep.columns:
    fig, ax = plt.subplots()
    ax.scatter(ep["wall_clock_seconds"], ep["sim_time_end"], c=ep["episode"], cmap="viridis", s=15)
    ax.set(xlabel="wall-clock seconds", ylabel="final sim time", title="Episode duration (real vs sim)")
    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label("episode")
    plt.show()
else:
    print("Wall-clock / sim-time columns not in this run.")

## Distribution of finished products / MTTR

In [ ]:
interest = [c for c in ["metric/finished_products", "metric/mttr",
                            "metric/fleet_availability_rate", "metric/throughput_rate"]
            if c in ep.columns and ep[c].notna().any()]
if interest:
    fig, axes = plt.subplots(1, len(interest), figsize=(4 * len(interest), 4))
    if len(interest) == 1: axes = [axes]
    for ax, col in zip(axes, interest):
        ep[col].dropna().plot.hist(ax=ax, bins=20)
        ax.set_title(col.split('/', 1)[-1])
    plt.tight_layout(); plt.show()